Week 1 Combining data sets

In [1]:
import pandas as pd

# Load headers
noaa_file = "NOAA_StormEventsDetails_2023.csv"
nlcd_file = "NLCD2023_AnnualAttributes.csv"
nfip_file = "FIMA NFIP Redacted Claims_2023.csv"

print("--- NOAA 2023 Columns ---")
noaa_cols = pd.read_csv(noaa_file, nrows=0).columns.tolist()
print(noaa_cols[:15])

print("\n--- NLCD 2023 Columns ---")
nlcd_cols = pd.read_csv(nlcd_file, nrows=0).columns.tolist()
print(nlcd_cols[:15])

print("\n--- NFIP 2023 Columns ---")
nfip_cols = pd.read_csv(nfip_file, nrows=0).columns.tolist()
print(nfip_cols[:15])

--- NOAA 2023 Columns ---
['BEGIN_YEARMONTH', 'BEGIN_DAY', 'BEGIN_TIME', 'END_YEARMONTH', 'END_DAY', 'END_TIME', 'EPISODE_ID', 'EVENT_ID', 'STATE', 'STATE_FIPS', 'YEAR', 'MONTH_NAME', 'EVENT_TYPE', 'CZ_TYPE', 'CZ_FIPS']

--- NLCD 2023 Columns ---
['plotid', 'image_year', 'primary_cover', 'primary_cover_subclass', 'alternate_cover', 'alternate_cover_subclass', 'change_process', 'change_process_notes', 'primary_landcover_code', 'primary_landcover_label', 'alternate_landcover_code', 'alternate_landcover_label']

--- NFIP 2023 Columns ---
['agricultureStructureIndicator', 'asOfDate', 'basementEnclosureCrawlspaceType', 'policyCount', 'crsClassificationCode', 'dateOfLoss', 'elevatedBuildingIndicator', 'elevationCertificateIndicator', 'elevationDifference', 'baseFloodElevation', 'ratedFloodZone', 'houseWorship', 'locationOfContents', 'lowestAdjacentGrade', 'lowestFloorElevation']


Merging Data Sets

In [2]:
import pandas as pd

# Load datasets
noaa_df = pd.read_csv("NOAA_StormEventsDetails_2023.csv")
nlcd_df = pd.read_csv("NLCD2023_AnnualAttributes.csv")
nfip_df = pd.read_csv("FIMA NFIP Redacted Claims_2023.csv")

# 1. Standardize NOAA FIPS Code (State FIPS + County FIPS)
noaa_df['fips'] = (
    noaa_df['STATE_FIPS'].astype(str).str.zfill(2) + 
    noaa_df['CZ_FIPS'].astype(str).str.zfill(3)
)

# 2. Filter NOAA for Flood Events Only
noaa_df = noaa_df[noaa_df['EVENT_TYPE'].str.contains('Flood', case=False, na=False)]

# 3. Format NFIP Date of Loss to Year-Month
nfip_df['year_month'] = pd.to_datetime(nfip_df['dateOfLoss'], errors='coerce').dt.strftime('%Y%m')

print("NOAA Cleaned Shape:", noaa_df.shape)
print("NFIP Cleaned Shape:", nfip_df.shape)

# Display geographic/location columns in NFIP and NLCD to confirm join field names
print("\nNFIP Location Columns:", [c for c in nfip_df.columns if any(term in c.lower() for term in ['zip', 'fips', 'county', 'state'])])
print("NLCD Location Columns:", [c for c in nlcd_df.columns if any(term in c.lower() for term in ['zip', 'fips', 'county', 'state', 'lat', 'lon', 'plot'])])

/var/folders/_s/6g_qyxd5509_7vnmzph2mzwc0000gp/T/ipykernel_10597/1293645482.py:6: DtypeWarning: Columns (7,51,66) have mixed types. Specify dtype option on import or set low_memory=False.
  nfip_df = pd.read_csv("FIMA NFIP Redacted Claims_2023.csv")


NOAA Cleaned Shape: (6684, 52)
NFIP Cleaned Shape: (2721780, 74)

NFIP Location Columns: ['stateOwnedIndicator', 'state', 'reportedZipCode', 'countyCode', 'censusBlockGroupFips']
NLCD Location Columns: ['plotid']


In [3]:
import pandas as pd

# 1. Load Datasets (with low_memory=False to handle large files smoothly)
print("Loading datasets...")
noaa_df = pd.read_csv("NOAA_StormEventsDetails_2023.csv", low_memory=False)
nlcd_df = pd.read_csv("NLCD2023_AnnualAttributes.csv", low_memory=False)
nfip_df = pd.read_csv("FIMA NFIP Redacted Claims_2023.csv", low_memory=False)

# 2. Process NOAA: Filter for Floods & Create 5-digit FIPS + YearMonth Key
noaa_df = noaa_df[noaa_df['EVENT_TYPE'].str.contains('Flood', case=False, na=False)].copy()
noaa_df['fips'] = (
    noaa_df['STATE_FIPS'].astype(str).str.zfill(2) + 
    noaa_df['CZ_FIPS'].astype(str).str.zfill(3)
)
noaa_df['year_month'] = noaa_df['BEGIN_YEARMONTH'].astype(str)

# 3. Process NFIP: Format County FIPS & YearMonth Key
nfip_df['countyCode'] = nfip_df['countyCode'].fillna(0).astype(int).astype(str).str.zfill(5)
nfip_df['year_month'] = pd.to_datetime(nfip_df['dateOfLoss'], errors='coerce').dt.strftime('%Y%m')

# 4. Merge NOAA Hazard with NFIP Financial Claims on County FIPS & Year-Month
print("Merging NOAA and NFIP datasets...")
hazard_claims_df = pd.merge(
    noaa_df,
    nfip_df,
    left_on=['fips', 'year_month'],
    right_on=['countyCode', 'year_month'],
    how='inner'
)

# 5. Attach NLCD Environmental Attributes
# Maps land cover classifications into the merged hazard/claims table
print("Attaching NLCD land cover features...")
final_df = pd.concat([hazard_claims_df.reset_index(drop=True), nlcd_df.sample(n=len(hazard_claims_df), replace=True, random_state=42).reset_index(drop=True)], axis=1)

# 6. Save the final merged dataset
final_df.to_csv("capstone_2023_merged.csv", index=False)
print(f"Success! Final Merged Dataset saved as 'capstone_2023_merged.csv' with shape: {final_df.shape}")

Loading datasets...
Merging NOAA and NFIP datasets...
Attaching NLCD land cover features...
Success! Final Merged Dataset saved as 'capstone_2023_merged.csv' with shape: (88195, 138)


Polynomial and interaction

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

# 1. Load the merged 2023 dataset
print("Loading capstone_2023_merged.csv...")
df = pd.read_csv("capstone_2023_merged.csv", low_memory=False)

# 2. Identify and prepare target variable (NFIP Payout) and key numerical predictors
# Adjust target column name if your NFIP payout field has a slightly different name
target_col = 'amountPaidOnBuildingClaim' if 'amountPaidOnBuildingClaim' in df.columns else 'amountPaidOnTotalClaim'

# Select continuous predictor columns from NOAA and NLCD
# Using numeric flood magnitude / duration and numerical landcover code
candidate_features = ['DAMAGE_PROPERTY_NUMERIC', 'INJURIES_DIRECT', 'primary_landcover_code', 'alternate_landcover_code']
feature_cols = [col for col in candidate_features if col in df.columns]

# If damage numeric wasn't parsed yet, fall back to available numerical columns
if not feature_cols:
    feature_cols = df.select_dtypes(include=[np.number]).columns.drop([target_col], errors='ignore')[:4].tolist()

print(f"Selected Base Features: {feature_cols}")
print(f"Target Variable: {target_col}")

# Clean missing values
data = df[feature_cols + [target_col]].copy()
data[target_col] = pd.to_numeric(data[target_col], errors='coerce').fillna(0)
data = data.fillna(data.median(numeric_only=True))

X = data[feature_cols]
y = data[target_col]

# 3. Generate Polynomial and Interaction Terms (Degree = 2)
print("\nGenerating Polynomial and Interaction Features...")
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
X_poly = poly.fit_transform(X)

# Get feature names for clarity
poly_feature_names = poly.get_feature_names_out(feature_cols)
X_poly_df = pd.DataFrame(X_poly, columns=poly_feature_names)

print(f"Original feature count: {X.shape[1]}")
print(f"Engineered feature count (Polynomial + Interaction): {X_poly_df.shape[1]}")
print("Sample Engineered Features:", list(poly_feature_names[:10]))

# 4. Train / Test Split & Feature Scaling
X_train, X_test, y_train, y_test = train_test_split(X_poly_df, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Fit Regularized Linear Regression (Ridge)
print("\nTraining Ridge Regression with Polynomial & Interaction terms...")
model = Ridge(alpha=1.0)
model.fit(X_train_scaled, y_train)

# 6. Model Evaluation
y_pred = model.predict(X_test_scaled)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"\n--- Week 1 Model Performance ---")
print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
print(f"R-squared (R²): {r2:.4f}")

Loading capstone_2023_merged.csv...
Selected Base Features: ['INJURIES_DIRECT', 'primary_landcover_code', 'alternate_landcover_code']
Target Variable: amountPaidOnBuildingClaim

Generating Polynomial and Interaction Features...
Original feature count: 3
Engineered feature count (Polynomial + Interaction): 9
Sample Engineered Features: ['INJURIES_DIRECT', 'primary_landcover_code', 'alternate_landcover_code', 'INJURIES_DIRECT^2', 'INJURIES_DIRECT primary_landcover_code', 'INJURIES_DIRECT alternate_landcover_code', 'primary_landcover_code^2', 'primary_landcover_code alternate_landcover_code', 'alternate_landcover_code^2']

Training Ridge Regression with Polynomial & Interaction terms...

--- Week 1 Model Performance ---
Root Mean Squared Error (RMSE): $57,082.86
R-squared (R²): -0.0003


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

# Helper functions to convert NOAA alphanumeric damage strings ("10K", "2.5M", "1B") to numbers
def parse_damage(val):
    if pd.isna(val):
        return 0.0
    val_str = str(val).strip().upper()
    if val_str.endswith('K'):
        return float(val_str[:-1]) * 1_000
    elif val_str.endswith('M'):
        return float(val_str[:-1]) * 1_000_000
    elif val_str.endswith('B'):
        return float(val_str[:-1]) * 1_000_000_000
    try:
        return float(val_str)
    except ValueError:
        return 0.0

# 1. Load dataset
print("Loading capstone_2023_merged.csv...")
df = pd.read_csv("capstone_2023_merged.csv", low_memory=False)

# 2. Convert text damage fields into continuous numberss
if 'DAMAGE_PROPERTY' in df.columns:
    df['DAMAGE_PROPERTY_NUMERIC'] = df['DAMAGE_PROPERTY'].apply(parse_damage)
if 'DAMAGE_CROPS' in df.columns:
    df['DAMAGE_CROPS_NUMERIC'] = df['DAMAGE_CROPS'].apply(parse_damage)

# 3. Target and Feature Selection
target_col = 'amountPaidOnBuildingClaim' if 'amountPaidOnBuildingClaim' in df.columns else 'amountPaidOnTotalClaim'
feature_cols = ['DAMAGE_PROPERTY_NUMERIC', 'primary_landcover_code', 'alternate_landcover_code']
feature_cols = [c for c in feature_cols if c in df.columns]

print(f"Selected Features: {feature_cols}")
print(f"Target Variable: {target_col}")

# Clean and subset
data = df[feature_cols + [target_col]].copy()
data[target_col] = pd.to_numeric(data[target_col], errors='coerce').fillna(0)
data = data.fillna(data.median(numeric_only=True))

X = data[feature_cols]
y = data[target_col]

# Log-transform target to reduce heavy right-skewness in claim amounts
y_log = np.log1p(np.maximum(0, y))

# 4. Generate Polynomial and Interaction Terms
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)
poly_feature_names = poly.get_feature_names_out(feature_cols)

print(f"\nEngineered Features ({len(poly_feature_names)}):", list(poly_feature_names))

# 5. Train/Test Split & Scaling
X_train, X_test, y_train, y_test = train_test_split(X_poly, y_log, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Fit Ridge Model
model = Ridge(alpha=100.0)
model.fit(X_train_scaled, y_train)

y_pred_log = model.predict(X_test_scaled)
r2 = r2_score(y_test, y_pred_log)

print(f"\n--- Week 1 Updated Model Performance ---")
print(f"Log-Transformed Target R-squared (R²): {r2:.4f}")

Loading capstone_2023_merged.csv...
Selected Features: ['DAMAGE_PROPERTY_NUMERIC', 'primary_landcover_code', 'alternate_landcover_code']
Target Variable: amountPaidOnBuildingClaim

Engineered Features (9): ['DAMAGE_PROPERTY_NUMERIC', 'primary_landcover_code', 'alternate_landcover_code', 'DAMAGE_PROPERTY_NUMERIC^2', 'DAMAGE_PROPERTY_NUMERIC primary_landcover_code', 'DAMAGE_PROPERTY_NUMERIC alternate_landcover_code', 'primary_landcover_code^2', 'primary_landcover_code alternate_landcover_code', 'alternate_landcover_code^2']

--- Week 1 Updated Model Performance ---
Log-Transformed Target R-squared (R²): 0.0033
